In [ ]:
import os
import numpy as np
import tifffile as tiff
from pathlib import Path
from tqdm import tqdm

def normalize_tif_batch(input_folder, output_folder='normalized_tif', 
                       lower_pct=1, upper_pct=99):
    """
    Batch normalize TIF images (excluding pixels with intensity 0, output floating point numbers in the range 0-1)
    
    Parameters:
    -----------
    input_folder : str
        Input folder path
    output_folder : str
        Output folder name
    lower_pct : float
        Lower percentile
    upper_pct : float
        Upper percentile
    """
    input_path = Path(input_folder)
    output_path = input_path / output_folder
    output_path.mkdir(exist_ok=True)
    
    # Get all TIF files
    tif_files = list(input_path.glob('*.tif')) + list(input_path.glob('*.tiff'))
    tif_files += list(input_path.glob('*.TIF')) + list(input_path.glob('*.TIFF'))
    
    print(f"Found {len(tif_files)} TIF files")
    print(f"Output format: 32-bit float (range 0-1)")
    
    for tif_path in tqdm(tif_files, desc="Normalization processing"):
        try:
            # Read TIF
            img = tiff.imread(tif_path)
            
            # Convert to float32
            img_float = img.astype(np.float32)
            
            # Exclude pixels with intensity 0
            non_zero_mask = img_float > 0

            # at this point non_zero_mask is a boolean matrix, say 2048 x 2048.
            non_zero_pixels = img_float[non_zero_mask]
            # non_zero_pixels is a 1D array that contains the pixel values of the non-zero pixels
            if len(non_zero_pixels) == 0:
                # Completely black image
                output_array = np.zeros_like(img_float, dtype=np.float32)
            else:
                # Calculate percentiles based on non-zero pixels
                robust_min = np.percentile(non_zero_pixels, lower_pct)
                robust_max = np.percentile(non_zero_pixels, upper_pct)
                
                # Normalize to [0, 1]
                output_array = np.zeros_like(img_float, dtype=np.float32)
                
                if robust_max > robust_min:
                    normalized = (non_zero_pixels - robust_min) / (robust_max - robust_min)
                    output_array[non_zero_mask] = np.clip(normalized, 0, 1)
                else:
                    output_array[non_zero_mask] = 0.5
            
            # Save as 16-bit float TIF
            output_filename = f"{tif_path.stem}_normalized.tif"
            output_file = output_path / output_filename
            tiff.imwrite(output_file, output_array)
            
        except Exception as e:
            print(f"\nError processing {tif_path.name}: {e}")
    
    print(f"\nComplete! Normalized images saved to: {output_path}")

if __name__ == "__main__":
    folder = "/mnt/HDD16TB/LanceKam_Lab/Daizong/Project/DLBCL/DLBCL/DLBCL_processed/01-03-2026 DLBCL 109241/formatted_actin"
    normalize_tif_batch(folder)